In [1]:
!pip install pandas numpy scikit-learn joblib

In [2]:
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [11]:
df = pd.read_csv("ResearchInformation3.csv")
df 

,Department,Gender,HSC,SSC,Income,Hometown,Computer,Preparation,Gaming,Attendance,Job,English,Extra,Semester,Last,Overall
0,Business Administration,Male,4.17,4.84,"Low (Below 15,000)",Village,3,More than 3 Hours,0-1 Hour,80%-100%,No,3,Yes,6th,3.220,3.350
1,Business Administration,Female,4.92,5.00,"Upper middle (30,000-50,000)",City,3,0-1 Hour,0-1 Hour,80%-100%,No,3,Yes,7th,3.467,3.467
2,Business Administration,Male,5.00,4.83,"Lower middle (15,000-30,000)",Village,3,0-1 Hour,More than 3 Hours,80%-100%,No,4,Yes,3rd,4.000,3.720
3,Business Administration,Male,4.00,4.50,"High (Above 50,000)",City,5,More than 3 Hours,More than 3 Hours,80%-100%,No,5,Yes,4th,3.800,3.750
4,Business Administration,Female,2.19,3.17,"Lower middle (15,000-30,000)",Village,3,0-1 Hour,2-3 Hours,80%-100%,No,3,Yes,4th,3.940,3.940
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
488,Public Health,Male,3.50,4.00,"Lower middle (15,000-30,000)",Village,2,More than 3 Hours,2-3 Hours,40%-59%,Yes,2,Yes,7th,3.250,3.100
489,Public Health,Male,4.50,4.00,"Upper middle (30,000-50,000)",City,4,2-3 Hours,2-3 Hours,60%-79%,No,3,Yes,9th,3.500,3.670
490,Sociology,Female,4.56,3.67,"Lower middle (15,000-30,000)",City,3,0-1 Hour,2-3 Hours,40%-59%,No,2,No,2nd,3.560,3.560
491,Sociology,Male,4.23,4.50,"Low (Below 15,000)",Village,3,More than 3 Hours,2-3 Hours,80%-100%,No,3,Yes,10th,3.610,3.750


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 493 entries, 0 to 492
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Department   493 non-null    str    
 1   Gender       493 non-null    str    
 2   HSC          493 non-null    float64
 3   SSC          493 non-null    float64
 4   Income       493 non-null    str    
 5   Hometown     493 non-null    str    
 6   Computer     493 non-null    int64  
 7   Preparation  493 non-null    str    
 8   Gaming       493 non-null    str    
 9   Attendance   493 non-null    str    
 10  Job          493 non-null    str    
 11  English      493 non-null    int64  
 12  Extra        493 non-null    str    
 13  Semester     493 non-null    str    
 14  Last         493 non-null    float64
 15  Overall      493 non-null    float64
dtypes: float64(4), int64(2), str(10)
memory usage: 61.8 KB


In [17]:
df.columns

Index(['Department', 'Gender', 'HSC', 'SSC', 'Income', 'Hometown', 'Computer',
       'Preparation', 'Gaming', 'Attendance', 'Job', 'English', 'Extra',
       'Semester', 'Last', 'Overall'],
      dtype='str')

In [20]:

print("from typing import Literal")
print("from pydantic import BaseModel, Field, ConfigDict\n")
print("class Feature(BaseModel):")
print("    model_config = ConfigDict(extra='forbid')\n")

for col in df.columns:
    # Skip 'Overall' if that's your target label variable (y) instead of an input feature (X)
    if col == 'Overall':
        continue
        
    dtype_name = str(df[col].dtype)
    unique_vals = df[col].dropna().unique().tolist()
    
    # 1. CATCHES YOUR STR COLUMNS PERFECTLY NOW
    if dtype_name == 'str' or dtype_name == 'object' or dtype_name == 'category':
        # Format strings into a clean Pydantic Literal choice array
        literal_options = ", ".join([f'"{val}"' for val in unique_vals])
        print(f'    {col}: Literal[{literal_options}]')
        
    # 2. Encoded numeric category code logic
    elif 'int' in dtype_name and len(unique_vals) <= 10:
        literal_options = ", ".join([str(val) for val in sorted(unique_vals)])
        print(f'    {col}: Literal[{literal_options}]')
        
    # 3. Continuous floating point features
    elif 'float' in dtype_name:
        min_val = round(float(df[col].min()), 2)
        max_val = round(float(df[col].max()), 2)
        print(f'    {col}: float = Field(..., ge={min_val}, le={max_val})')
        
    # 4. Standard integers
    elif 'int' in dtype_name:
        min_val = int(df[col].min())
        max_val = int(df[col].max())
        print(f'    {col}: int = Field(..., ge={min_val}, le={max_val})')


from typing import Literal
from pydantic import BaseModel, Field, ConfigDict

class Feature(BaseModel):
    model_config = ConfigDict(extra='forbid')

    Department: Literal["Business Administration", "Computer Science and Engineering", "Economics", "Electrical and Electronic Engineering", "English", "Journalism, Communication and Media Studies", "Law and Human Rights", "Political Science", "Public Health", "Sociology"]
    Gender: Literal["Male", "Female"]
    HSC: float = Field(..., ge=2.17, le=5.0)
    SSC: float = Field(..., ge=3.0, le=5.0)
    Income: Literal["Low (Below 15,000)", "Upper middle (30,000-50,000)", "Lower middle (15,000-30,000)", "High (Above 50,000)", "Low (Below 15,000) ", "Lower middle (15,000-30,000) ", "High (Above 50,000) ", "Lower middle (15,000-30,000)  ", "High (Above 50,000)  ", "Upper middle (30,000-50,000) "]
    Hometown: Literal["Village", "City"]
    Computer: Literal[1, 2, 3, 4, 5]
    Preparation: Literal["More than 3 Hours", "0-1 Hour", "2-3 Hours"

In [5]:
df.describe()

,HSC,SSC,Computer,English,Last,Overall
count,493.000000,493.000000,493.000000,493.000000,493.000000,493.000000
mean,4.156775,4.767647,3.338742,3.569980,3.163888,3.188383
std,0.546504,0.349538,1.252988,0.870515,0.641475,0.591801
min,2.170000,3.000000,1.000000,1.000000,1.000000,1.000000
25%,3.830000,4.680000,3.000000,3.000000,2.810000,2.880000
50%,4.170000,4.940000,3.000000,4.000000,3.250000,3.270000
75%,4.500000,5.000000,4.000000,4.000000,3.670000,3.680000
max,5.000000,5.000000,5.000000,5.000000,4.000000,4.000000


In [6]:
X = df.drop("Overall", axis=1)
y = df["Overall"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
numeric_cols = ["HSC", "SSC", "Computer", "English", "Last"]
cat_cols = ["Department", "Gender", "Income", "Hometown", "Preparation", "Gaming", "Attendance", "Job", "Extra", "Semester"]

num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", missing_values="Missing")),
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessing_stage = ColumnTransformer(transformers=[
    ("num_preprocess", num_transformer, numeric_cols),
    ("cat_preprocessing", cat_transformer, cat_cols)
])

full_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessing_stage),
    ("model_stage", Ridge())
])

model = full_pipeline.fit(X_train, y_train)

In [8]:
list(X_train.columns)

['Department',
 'Gender',
 'HSC',
 'SSC',
 'Income',
 'Hometown',
 'Computer',
 'Preparation',
 'Gaming',
 'Attendance',
 'Job',
 'English',
 'Extra',
 'Semester',
 'Last']

In [9]:
model_bundle = {
    "pipeline": full_pipeline,
    "metadata": {
        "version": 1.0,
        "features": list(X_train.columns),
        "target": "Overall",
        "classification_threshold": 0.5
    }
}

joblib.dump(model_bundle, "ridge_model.joblib")
print("model saved successfully")

model saved successfully
